In [ ]:
import torch
import torch.nn as nn
from collections import deque

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

relu = nn.ReLU()

def relu_derivative(x):
    return (x > 0).float()

In [ ]:
def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    pos = torch.arange(0, max_seq_len).unsqueeze(1)
    i = torch.arange(0, d_model, 2)
    div = 10000 ** (i / d_model)
    PE[:, 0::2] = torch.sin(pos / div)
    PE[:, 1::2] = torch.cos(pos / div)
    return PE

In [ ]:
class Attention_Block(nn.Module):
  def __init__(self, d_model:int, num_heads:int):
    super().__init__()
    self.num_heads = num_heads
    self.input = None
    self.Exp = None
    self.d_k = d_model // num_heads
    self.d_model = d_model
    self.Q_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.K_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.V_M = nn.Parameter(torch.randn(num_heads, d_model, self.d_k))
    self.W_O = nn.Parameter(torch.randn(d_model, d_model))
    self.Q = None
    self.K = None
    self.V = None
    self.scores = None
    self.A = None
    self.deltaE_heads = None
    self.deltaE_cat = None

  def forward_pass(self, E, mask=False):
    n = E.shape[0]
    self.inputs = E
    self.Exp = self.inputs.unsqueeze(0).expand(self.num_heads, -1, -1)
    self.Q = torch.bmm(self.Exp, self.Q_M)
    self.K = torch.bmm(self.Exp, self.K_M)
    self.V = torch.bmm(self.Exp, self.V_M)

    self.scores = torch.bmm(self.Q, self.K.transpose(-2, -1)) / (self.d_k) ** 0.5
    if mask:
      m = torch.triu(torch.ones(n, n), diagonal=1).bool().to(self.Q_M.device)
      self.scores = self.scores.masked_fill(m, float('-inf')).to(device)

    self.A = torch.softmax(self.scores, dim=-1)
    self.deltaE_heads = torch.bmm(self.A, self.V)
    self.deltaE_cat = self.deltaE_heads.transpose(0, 1).reshape(n, -1)

    return self.deltaE_cat @ self.W_O

  def backpropagation(self, i_g):
    n = self.inputs.shape[0]

    dW_O = self.deltaE_cat.T @ i_g
    d_deltaE_cat = i_g @ self.W_O.T

    # split gradient back to each head
    d_deltaE_heads = d_deltaE_cat.reshape(n, self.num_heads, self.d_k).transpose(0, 1)

    dA = torch.bmm(d_deltaE_heads, self.V.transpose(-2, -1))
    dV = torch.bmm(self.A.transpose(-2, -1), d_deltaE_heads)

    # softmax jacobian
    dS = self.A * (dA - (dA * self.A).sum(dim=-1, keepdim=True))

    dQ = torch.bmm(dS, self.K) / self.d_k ** 0.5
    dK = torch.bmm(dS.transpose(-2, -1), self.Q) / self.d_k ** 0.5

    dW_Q = torch.bmm(self.Exp.transpose(-2, -1), dQ)
    dW_K = torch.bmm(self.Exp.transpose(-2, -1), dK)
    dW_V = torch.bmm(self.Exp.transpose(-2, -1), dV)

    # sum gradients across all heads to get dX
    dX = (torch.bmm(dQ, self.Q_M.transpose(-2, -1)) +
          torch.bmm(dK, self.K_M.transpose(-2, -1)) +
          torch.bmm(dV, self.V_M.transpose(-2, -1))).sum(dim=0)

    return dW_Q, dW_K, dW_V, dW_O, dX


class Layer(nn.Module):
  def __init__(self, input, prev):
    super().__init__()
    self.weights = nn.Parameter(torch.randn(prev, input))
    self.bias = nn.Parameter(torch.randn(input))
    self.activations = None
    self.z = None
    self.inputs = input
    self.prev = prev


class MLP(nn.Module):
  def __init__(self, num_layers:int, layers:list):
    super().__init__()
    self.num = num_layers
    self.layers = nn.ModuleList()
    self.l = layers
    self.input = None
    for i in range(1, self.num + 1):
      self.layers.append(Layer(layers[i], layers[i - 1]))

  def forward_pass(self, input):
    self.layers[0].activations = input
    self.input = input
    for i in range(1, self.num):
      self.layers[i].z = (self.layers[i-1].activations @ self.layers[i].weights) + self.layers[i].bias
      self.layers[i].activations = relu(self.layers[i].z)

    last = self.num
    self.layers[last].z = self.layers[last-1].activations @ self.layers[last].weights + self.layers[last].bias
    self.layers[last].activations = self.layers[last].z

    return self.layers[self.num].activations

  def backpropagation(self, i_g):
    dW = deque()
    db = deque()
    delta = i_g
    for i in range(self.num, 0, -1):
      if i != self.num:
        delta = delta @ self.layers[i+1].weights.T * relu_derivative(self.layers[i].z)
        dW.appendleft(self.layers[i-1].activations.T @ delta)
      else:
        delta = i_g
        dW.appendleft(self.layers[i-1].activations.T @ delta)
      db.appendleft(delta)

    return dW, db, delta @ self.layers[1].weights.T


class LayerNorm(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.input = None
        self.mu = None
        self.var = None
        self.x_hat = None

    def forward_pass(self, E):
        self.input = E
        self.mu = E.sum(dim=-1, keepdim=True) / self.d_model
        self.var = ((E - self.mu) ** 2).sum(dim=-1, keepdim=True) / self.d_model
        self.x_hat = (E - self.mu) / (self.var + 1e-5) ** 0.5
        return self.gamma * self.x_hat + self.beta

    def backpropagation(self, i_g):
        sigma = (self.var + 1e-5) ** 0.5

        d_gamma = (i_g * self.x_hat).sum(dim=0)
        d_beta = i_g.sum(dim=0)

        dx_hat = i_g * self.gamma

        # quotient rule accounting for mu and sigma both depending on X
        dX = (self.gamma / sigma) * (
            dx_hat
            - dx_hat.sum(dim=-1, keepdim=True) / self.d_model
            - self.x_hat * (dx_hat * self.x_hat).sum(dim=-1, keepdim=True) / self.d_model
        )

        return d_gamma, d_beta, dX


class Transformer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attention = Attention_Block(d_model, num_heads).to(device)
        self.mlp = MLP(3, [d_model, 4*d_model, 4*d_model, d_model]).to(device)
        self.norm1 = LayerNorm(d_model).to(device)
        self.norm2 = LayerNorm(d_model).to(device)
        self.input = None
        self.norm1_out = None
        self.attn_out = None
        self.norm2_out = None
        self.mlp_out = None

    def forward_pass(self, E, mask=False):
        self.input = E

        self.norm1_out = self.norm1.forward_pass(E)
        self.attn_out = self.attention.forward_pass(self.norm1_out, mask)
        E = E + self.attn_out

        self.norm2_out = self.norm2.forward_pass(E)
        self.mlp_out = self.mlp.forward_pass(self.norm2_out)
        E = E + self.mlp_out

        return E

    def backpropagation(self, i_g):
        # residual: gradient copies to both branches, then sums when they rejoin
        d_mlp_out = i_g
        d_E_mid = i_g

        dW_mlp, db_mlp, d_norm2_out = self.mlp.backpropagation(d_mlp_out)
        d_gamma2, d_beta2, d_norm2_in = self.norm2.backpropagation(d_norm2_out)

        d_E_mid = d_E_mid + d_norm2_in

        d_attn_out = d_E_mid
        d_input = d_E_mid

        dW_Q, dW_K, dW_V, dW_O, d_norm1_out = self.attention.backpropagation(d_attn_out)
        d_gamma1, d_beta1, d_norm1_in = self.norm1.backpropagation(d_norm1_out)

        d_input = d_input + d_norm1_in

        grads = {
            'dW_Q': dW_Q, 'dW_K': dW_K, 'dW_V': dW_V, 'dW_O': dW_O,
            'dW_mlp': dW_mlp, 'db_mlp': db_mlp,
            'd_gamma1': d_gamma1, 'd_beta1': d_beta1,
            'd_gamma2': d_gamma2, 'd_beta2': d_beta2,
        }

        return grads, d_input


class LLM(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_blocks, max_seq_len):
        super().__init__()
        self.embedding = nn.Parameter(torch.randn(vocab_size, d_model))
        self.pos_encoding = positional_encoding(max_seq_len, d_model).to(device)
        self.blocks = nn.ModuleList([
            Transformer(d_model, num_heads)
            for _ in range(num_blocks)
        ])
        self.unembedding = nn.Parameter(torch.randn(d_model, vocab_size))
        self.token_ids = None
        self.E = None
        self.logits = None
        self.probs = None

    def forward_pass(self, token_ids, mask=False):
        self.token_ids = token_ids
        self.E = self.embedding[token_ids] + self.pos_encoding[:len(token_ids)]
        for block in self.blocks:
            self.E = block.forward_pass(self.E, mask)
        self.logits = self.E[-1] @ self.unembedding
        self.probs = torch.softmax(self.logits, dim=-1)
        return self.probs

    def loss(self, target_id):
        return -torch.log(self.probs[target_id] + 1e-9)

    def backpropagation(self, target_id):
        all_grads = []

        # combined cross-entropy + softmax gradient = probs - one_hot
        d_logits = self.probs.clone()
        d_logits[target_id] -= 1.0

        dW_unembed = self.E[-1].unsqueeze(1) @ d_logits.unsqueeze(0)
        d_E_last = d_logits @ self.unembedding.T

        # only last token has gradient from unembedding
        d_E = torch.zeros_like(self.E)
        d_E[-1] = d_E_last

        for block in reversed(self.blocks):
            grads, d_E = block.backpropagation(d_E)
            all_grads.append(grads)

        # scatter gradients back to embedding rows
        d_embedding = torch.zeros_like(self.embedding)
        for t, token_id in enumerate(self.token_ids):
            d_embedding[token_id] += d_E[t]

        return all_grads, dW_unembed, d_embedding